In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from collections import Counter
import re
import string

In [2]:
df = pd.read_csv('reviews_disneyhotstar.csv')

print("Jumlah baris dan kolom:")
print(df.shape)

Jumlah baris dan kolom:
(15000, 11)


In [3]:
df = pd.read_csv('reviews_disneyhotstar.csv')

print("\nInformasi kolom:")
print(df.info())


Informasi kolom:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              15000 non-null  object
 1   userName              15000 non-null  object
 2   userImage             15000 non-null  object
 3   content               15000 non-null  object
 4   score                 15000 non-null  int64 
 5   thumbsUpCount         15000 non-null  int64 
 6   reviewCreatedVersion  10110 non-null  object
 7   at                    15000 non-null  object
 8   replyContent          3954 non-null   object
 9   repliedAt             3954 non-null   object
 10  appVersion            10110 non-null  object
dtypes: int64(2), object(9)
memory usage: 1.3+ MB
None


In [4]:
df = pd.read_csv('reviews_disneyhotstar.csv')

print("\n5 baris pertama dataset:")
print(df.head())


5 baris pertama dataset:
                               reviewId         userName  \
0  9337808f-aa37-428a-98a7-e9ccc4f63b19  Pengguna Google   
1  02c74257-a1eb-4c67-8e71-5993e6044e2f  Pengguna Google   
2  1a1ff5de-5804-4e05-8e1a-3c2133a2258b  Pengguna Google   
3  2cc6ff51-6c16-45a1-b1db-989103545359  Pengguna Google   
4  48491ffb-8926-4dfe-8db9-08698cd99393  Pengguna Google   

                                           userImage  \
0  https://play-lh.googleusercontent.com/EGemoI2N...   
1  https://play-lh.googleusercontent.com/EGemoI2N...   
2  https://play-lh.googleusercontent.com/EGemoI2N...   
3  https://play-lh.googleusercontent.com/EGemoI2N...   
4  https://play-lh.googleusercontent.com/EGemoI2N...   

                                             content  score  thumbsUpCount  \
0  Ga ikhlas sumpah. bakal gw tagih di akhirat. b...      1              3   
1                                    Banyak bugnya 😢      1              0   
2                           Jelek banget a

In [5]:
# Menangani data yang hilang pada kolom 'username'
df_cleaned = df.dropna(subset=['userName'])
print(df_cleaned.isnull().sum())

reviewId                    0
userName                    0
userImage                   0
content                     0
score                       0
thumbsUpCount               0
reviewCreatedVersion     4890
at                          0
replyContent            11046
repliedAt               11046
appVersion               4890
dtype: int64


In [6]:
# Fungsi untuk membersihkan teks
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

df['cleaned_content'] = df['content'].apply(clean_text)

In [7]:
def get_sentiment(text):
    from textblob import TextBlob
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0:
        return 'positif'
    elif polarity < 0:
        return 'negatif'
    else:
        return 'netral'

df['label'] = df['cleaned_content'].apply(get_sentiment)

In [8]:
# Encode label
y = LabelEncoder().fit_transform(df['label'])
X = df['cleaned_content']

# Pisahkan data menjadi train dan test
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
# Preprocessing untuk model berbasis TF-IDF
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train_raw)
X_test_tfidf = vectorizer.transform(X_test_raw)

smote = SMOTE(random_state=42)
X_train_tfidf_smote, y_train_smote = smote.fit_resample(X_train_tfidf, y_train)

print("Distribusi label setelah SMOTE:", Counter(y_train_smote))

Distribusi label setelah SMOTE: Counter({np.int64(1): 10514, np.int64(2): 10514, np.int64(0): 10514})


In [10]:
# 1. Model SVM
svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train_tfidf_smote, y_train_smote)
y_pred_svm = svm.predict(X_test_tfidf)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("\nEvaluasi Model SVM:")
print(f"Accuracy: {accuracy_svm}")
print(classification_report(y_test, y_pred_svm, target_names=['negatif', 'netral', 'positif']))


Evaluasi Model SVM:
Accuracy: 0.977
              precision    recall  f1-score   support

     negatif       0.91      0.76      0.83       105
      netral       0.98      1.00      0.99      2629
     positif       0.97      0.86      0.91       266

    accuracy                           0.98      3000
   macro avg       0.95      0.87      0.91      3000
weighted avg       0.98      0.98      0.98      3000



In [11]:
# 2. Model Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_tfidf_smote, y_train_smote)
y_pred_rf = rf.predict(X_test_tfidf)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("\nEvaluasi Model Random Forest:")
print(f"Accuracy: {accuracy_rf}")
print(classification_report(y_test, y_pred_rf, target_names=['negatif', 'netral', 'positif']))


Evaluasi Model Random Forest:
Accuracy: 0.948
              precision    recall  f1-score   support

     negatif       0.93      0.53      0.68       105
      netral       0.95      1.00      0.97      2629
     positif       0.94      0.63      0.76       266

    accuracy                           0.95      3000
   macro avg       0.94      0.72      0.80      3000
weighted avg       0.95      0.95      0.94      3000



In [12]:

# Preprocessing untuk model LSTM
max_words = 10000
max_len = 100
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train_raw)

X_train_seq = tokenizer.texts_to_sequences(X_train_raw)
X_test_seq = tokenizer.texts_to_sequences(X_test_raw)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

# Konversi label ke one-hot encoding
y_train_cat = pd.get_dummies(y_train).values
y_test_cat = pd.get_dummies(y_test).values

In [13]:
# 3. Model LSTM
model_lstm = Sequential([
    Embedding(input_dim=max_words, output_dim=100, input_length=max_len),
    LSTM(128, return_sequences=True, kernel_regularizer='l2'),
    Dropout(0.5),
    LSTM(64, kernel_regularizer='l2'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model_lstm.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2)
]

history = model_lstm.fit(
    X_train_pad, y_train_cat,
    epochs=15,
    batch_size=64,
    validation_data=(X_test_pad, y_test_cat),
    callbacks=callbacks
)

oss, accuracy_lstm = model_lstm.evaluate(X_test_pad, y_test_cat)
print("\nEvaluasi Model LSTM:")
print(f"Accuracy: {accuracy_lstm}")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.8526 - loss: 2.0074 - val_accuracy: 0.8763 - val_loss: 0.4776 - learning_rate: 0.0010
Epoch 2/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.8810 - loss: 0.4668 - val_accuracy: 0.8763 - val_loss: 0.4540 - learning_rate: 0.0010
Epoch 3/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8755 - loss: 0.4663 - val_accuracy: 0.8763 - val_loss: 0.4488 - learning_rate: 0.0010
Epoch 4/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.8757 - loss: 0.4597 - val_accuracy: 0.8763 - val_loss: 0.4479 - learning_rate: 0.0010
Epoch 5/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8742 - loss: 0.4631 - val_accuracy: 0.8763 - val_loss: 0.4497 - learning_rate: 0.0010
Epoch 6/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.8790 - loss: 0.4503 - val_accuracy: 0.8763 - val_loss: 0.4487 - learning_rate: 0.0010
Epoch 7/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8787 - loss: 0

In [14]:
!pip freeze > requirements.txt
from google.colab import files
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>